In [1]:
import cudf 
import cudf as df
import cupy as cp
import numpy as np
import gc
import os
import time

from cuml.preprocessing import StandardScaler
from cuml.neighbors import NearestNeighbors

print("cuDF version :", cudf.__version__)
print("CuPy version :", cp.__version__)

try:
    import cuml
    print("cuML version :", cuml.__version__)
except Exception:
    print("cuML loaded")


cuDF version : 26.06.01
CuPy version : 14.2.0
cuML version : 26.06.00


In [2]:

print("=" * 70)
print("GPU CHECK")
print("=" * 70)

try:
    device = cp.cuda.Device()
    print("GPU ID:", device.id)

    free_mem, total_mem = cp.cuda.runtime.memGetInfo()

    print(
        f"GPU total memory : "
        f"{total_mem / (1024**3):.2f} GB"
    )

    print(
        f"GPU free memory  : "
        f"{free_mem / (1024**3):.2f} GB"
    )

except Exception as e:
    print("GPU check failed:")
    print(e)


GPU CHECK
GPU ID: 0
GPU total memory : 6.00 GB
GPU free memory  : 4.95 GB


In [3]:
start = time.time()

train_transaction = cudf.read_csv(
    r"/home/abhin/mainproject/FEDBANK/data/train_transaction.csv"
)

print("Transaction shape:", train_transaction.shape)
print("Time:", round(time.time() - start, 2), "seconds")

Transaction shape: (590540, 394)
Time: 1.42 seconds


In [4]:
train_identity=cudf.read_csv(r"/home/abhin/mainproject/FEDBANK/data/train_identity.csv")

In [5]:
start = time.time()

df = train_transaction.merge(
    train_identity,
    on="TransactionID",
    how="left"
)

print("Merged shape:", df.shape)
print("Time:", round(time.time() - start, 2), "seconds")

Merged shape: (590540, 434)
Time: 0.35 seconds


In [6]:
print("Number of rows   :", len(df))
print("Number of columns:", len(df.columns))

print("\nFirst 20 columns:")
print(df.columns[:20])

print("\nLast 20 columns:")
print(df.columns[-20:])

Number of rows   : 590540
Number of columns: 434

First 20 columns:
Index(['TransactionID', 'isFraud', 'TransactionDT', 'TransactionAmt',
       'ProductCD', 'card1', 'card2', 'card3', 'card4', 'card5', 'card6',
       'addr1', 'addr2', 'dist1', 'dist2', 'P_emaildomain', 'R_emaildomain',
       'C1', 'C2', 'C3'],
      dtype='object')

Last 20 columns:
Index(['id_21', 'id_22', 'id_23', 'id_24', 'id_25', 'id_26', 'id_27', 'id_28',
       'id_29', 'id_30', 'id_31', 'id_32', 'id_33', 'id_34', 'id_35', 'id_36',
       'id_37', 'id_38', 'DeviceType', 'DeviceInfo'],
      dtype='object')


In [7]:
print(df["isFraud"].value_counts())

fraud_count = int((df["isFraud"] == 1).sum())
normal_count = int((df["isFraud"] == 0).sum())

print("\nNormal transactions:", normal_count)
print("Fraud transactions :", fraud_count)

print(
    "Fraud percentage:",
    round((fraud_count / len(df)) * 100, 4),
    "%"
)

isFraud
0    569877
1     20663
Name: count, dtype: int64

Normal transactions: 569877
Fraud transactions : 20663
Fraud percentage: 3.499 %


In [8]:
null_report = cudf.DataFrame({
    "column": df.columns,
    "missing_count": [
        int(df[col].isna().sum())
        for col in df.columns
    ]
})

null_report["missing_percent"] = (
    null_report["missing_count"] / len(df) * 100
)

null_report = null_report.sort_values(
    "missing_percent",
    ascending=False
)

null_report.head(30)

,column,missing_count,missing_percent
417,id_24,585793,99.196159
418,id_25,585408,99.130965
400,id_07,585385,99.127070
401,id_08,585385,99.127070
414,id_21,585381,99.126393
419,id_26,585377,99.125715
415,id_22,585371,99.124699
416,id_23,585371,99.124699
420,id_27,585371,99.124699
14,dist2,552913,93.628374


In [9]:
v_features = [
    col for col in df.columns
    if col.startswith("V")
]

print("Number of V features:", len(v_features))

print(v_features[:20])

Number of V features: 339
['V1', 'V2', 'V3', 'V4', 'V5', 'V6', 'V7', 'V8', 'V9', 'V10', 'V11', 'V12', 'V13', 'V14', 'V15', 'V16', 'V17', 'V18', 'V19', 'V20']


In [10]:
v_report = cudf.DataFrame({
    "feature": v_features,
    "missing_count": [
        int(df[col].isna().sum())
        for col in v_features
    ]
})

v_report["missing_percent"] = (
    v_report["missing_count"] / len(df) * 100
)

v_report = v_report.sort_values(
    "missing_percent",
    ascending=False
)

v_report.head(30)

,feature,missing_count,missing_percent
137,V138,508595,86.123717
138,V139,508595,86.123717
139,V140,508595,86.123717
140,V141,508595,86.123717
141,V142,508595,86.123717
145,V146,508595,86.123717
146,V147,508595,86.123717
147,V148,508595,86.123717
148,V149,508595,86.123717
152,V153,508595,86.123717


In [11]:
# ============================================================
# FEATURE CATEGORIZATION
# ============================================================

very_high_missing = []
high_missing = []
knn_features = []
complete_features = []

for col in v_features:

    missing_percent = (
        df[col].isna().sum() / len(df) * 100
    )

    if missing_percent >= 90:
        very_high_missing.append(col)

    elif missing_percent >= 50:
        high_missing.append(col)

    elif missing_percent > 0:
        knn_features.append(col)

    else:
        complete_features.append(col)

print("Feature categorization")
print("=" * 50)
print(">= 90% missing :", len(very_high_missing))
print("50-90% missing :", len(high_missing))
print("< 50% missing  :", len(knn_features))
print("No missing     :", len(complete_features))
print("Total V        :", len(v_features))

Feature categorization
>= 90% missing : 0
50-90% missing : 159
< 50% missing  : 180
No missing     : 0
Total V        : 339


In [12]:
missing_indicator_features = []

for col in v_features:

    missing_percent = (
        df[col].isna().sum() / len(df) * 100
    )

    if missing_percent > 5:

        indicator_name = col + "_missing"

        df[indicator_name] = df[col].isna().astype("int8")

        missing_indicator_features.append(indicator_name)

print(
    "Created",
    len(missing_indicator_features),
    "missing indicators."
)

Created 253 missing indicators.


In [13]:
numeric_columns = df.select_dtypes(
    include=["float32", "float64", "int8", "int16", "int32", "int64"]
).columns

for col in numeric_columns:

    df[col] = df[col].replace(
        [cp.inf, -cp.inf],
        None
    )

print("Infinite-value cleanup completed.")

Infinite-value cleanup completed.


In [14]:
start = time.time()

median_imputed_features = (
    very_high_missing + high_missing
)

for col in median_imputed_features:

    median_value = df[col].median()

    if median_value is not None:
        df[col] = df[col].fillna(median_value)

print(
    "Median imputation completed for",
    len(median_imputed_features),
    "features."
)

print("Time:", round(time.time() - start, 2), "seconds")

Median imputation completed for 159 features.
Time: 1.29 seconds


In [15]:
remaining_report = cudf.DataFrame({
    "feature": v_features,
    "missing_count": [
        int(df[col].isna().sum())
        for col in v_features
    ]
})

remaining_report["missing_percent"] = (
    remaining_report["missing_count"] / len(df) * 100
)

remaining_report = remaining_report.sort_values(
    "missing_percent",
    ascending=False
)

remaining_report.head(30)

,feature,missing_count,missing_percent
0,V1,279287,47.293494
1,V2,279287,47.293494
2,V3,279287,47.293494
3,V4,279287,47.293494
4,V5,279287,47.293494
5,V6,279287,47.293494
6,V7,279287,47.293494
7,V8,279287,47.293494
8,V9,279287,47.293494
9,V10,279287,47.293494


In [16]:
knn_feature_mask = df[knn_features].notna().all(axis=1)

complete_indices = df.index[knn_feature_mask]

print(
    "Complete reference rows:",
    len(complete_indices)
)

Complete reference rows: 254676


In [17]:
REFERENCE_SIZE = 20_000

if len(complete_indices) > REFERENCE_SIZE:

    reference_indices = complete_indices.to_series().sample(
        n=REFERENCE_SIZE,
        random_state=42
    ).index

else:

    reference_indices = complete_indices

print(
    "Reference rows selected:",
    len(reference_indices)
)

Reference rows selected: 20000


In [18]:
reference_df = df.loc[
    reference_indices,
    knn_features
].astype("float32")

print("Reference shape:", reference_df.shape)

Reference shape: (20000, 180)


In [19]:
reference_gpu = cp.asarray(
    reference_df.to_cupy()
)

print("Reference GPU shape:", reference_gpu.shape)
print("GPU dtype:", reference_gpu.dtype)

Reference GPU shape: (20000, 180)
GPU dtype: float32


In [20]:
reference_mean = cp.nanmean(
    reference_gpu,
    axis=0
)

print(
    "Reference mean shape:",
    reference_mean.shape
)

Reference mean shape: (180,)


In [21]:
reference_filled = cp.where(
    cp.isnan(reference_gpu),
    reference_mean,
    reference_gpu
)

print(
    "NaNs remaining in reference:",
    int(cp.isnan(reference_filled).sum())
)

NaNs remaining in reference: 0


In [22]:
scaler = StandardScaler()

reference_scaled = scaler.fit_transform(
    reference_filled
)

print(
    "Scaled reference shape:",
    reference_scaled.shape
)

Scaled reference shape: (20000, 180)


In [23]:
K = 5

knn = NearestNeighbors(
    n_neighbors=K
)

knn.fit(reference_scaled)

print("GPU KNN model ready.")

GPU KNN model ready.


In [24]:
missing_knn_mask = df[knn_features].isna().any(axis=1)

missing_knn_indices = df.index[missing_knn_mask]

print(
    "Rows requiring KNN imputation:",
    len(missing_knn_indices)
)

Rows requiring KNN imputation: 335864


In [25]:
def gpu_knn_impute_batch(
    batch_df,
    reference_gpu,
    reference_scaled,
    reference_mean,
    knn,
    scaler,
    k
):

    # Original batch values
    batch_gpu = cp.asarray(
        batch_df.to_cupy()
    ).astype(cp.float32)

    # Missing-value mask
    missing_mask = cp.isnan(batch_gpu)

    # Fill missing values temporarily for neighbor search
    batch_filled = cp.where(
        missing_mask,
        reference_mean,
        batch_gpu
    )

    # Scale batch
    batch_scaled = scaler.transform(
        batch_filled
    )

    # Find nearest neighbours
    distances, indices = knn.kneighbors(
        batch_scaled,
        n_neighbors=k
    )

    # Get original reference values
    neighbor_values = reference_gpu[
        indices
    ]

    # Mean value for each feature across neighbours
    neighbor_means = cp.mean(
        neighbor_values,
        axis=1
    )

    # Replace only missing values
    result = cp.where(
        missing_mask,
        neighbor_means,
        batch_gpu
    )

    return result

In [29]:
BATCH_SIZE = 1000

start = time.time()

total_rows = len(missing_knn_indices)

print("Rows to process:", total_rows)
print("KNN features:", len(knn_features))
print("Batch size:", BATCH_SIZE)
print("=" * 60)

for start_pos in range(0, total_rows, BATCH_SIZE):

    end_pos = min(
        start_pos + BATCH_SIZE,
        total_rows
    )

    batch_indices = missing_knn_indices[
        start_pos:end_pos
    ]

    # --------------------------------------------------------
    # Get current batch
    # --------------------------------------------------------

    batch_df = df.loc[
        batch_indices,
        knn_features
    ].astype("float32")

    # --------------------------------------------------------
    # GPU KNN imputation
    # --------------------------------------------------------

    batch_result = gpu_knn_impute_batch(
        batch_df=batch_df,
        reference_gpu=reference_gpu,
        reference_scaled=reference_scaled,
        reference_mean=reference_mean,
        knn=knn,
        scaler=scaler,
        k=K
    )

    # --------------------------------------------------------
    # Safety check
    # --------------------------------------------------------

    assert batch_result.shape == (
        len(batch_indices),
        len(knn_features)
    ), (
        f"Unexpected result shape: {batch_result.shape}, "
        f"expected ({len(batch_indices)}, {len(knn_features)})"
    )

    # --------------------------------------------------------
    # Assign column by column
    # Avoid cuDF multi-column .loc assignment issue
    # --------------------------------------------------------

    for col_idx, col in enumerate(knn_features):

        df.loc[
            batch_indices,
            col
        ] = batch_result[:, col_idx]

    # --------------------------------------------------------
    # Progress
    # --------------------------------------------------------

    if (
        start_pos % (BATCH_SIZE * 10) == 0
        or end_pos == total_rows
    ):

        percent = (
            end_pos / total_rows
        ) * 100

        print(
            f"Progress: {end_pos}/{total_rows} "
            f"({percent:.2f}%)"
        )

    # --------------------------------------------------------
    # Free memory
    # --------------------------------------------------------

    del batch_df
    del batch_result

    cp.get_default_memory_pool().free_all_blocks()
    gc.collect()


print("\nKNN imputation completed.")

print(
    "Time:",
    round(time.time() - start, 2),
    "seconds"
)

Rows to process: 335864
KNN features: 180
Batch size: 1000
Progress: 1000/335864 (0.30%)
Progress: 11000/335864 (3.28%)
Progress: 21000/335864 (6.25%)
Progress: 31000/335864 (9.23%)
Progress: 41000/335864 (12.21%)
Progress: 51000/335864 (15.18%)
Progress: 61000/335864 (18.16%)
Progress: 71000/335864 (21.14%)
Progress: 81000/335864 (24.12%)
Progress: 91000/335864 (27.09%)
Progress: 101000/335864 (30.07%)
Progress: 111000/335864 (33.05%)
Progress: 121000/335864 (36.03%)
Progress: 131000/335864 (39.00%)
Progress: 141000/335864 (41.98%)
Progress: 151000/335864 (44.96%)
Progress: 161000/335864 (47.94%)
Progress: 171000/335864 (50.91%)
Progress: 181000/335864 (53.89%)
Progress: 191000/335864 (56.87%)
Progress: 201000/335864 (59.85%)
Progress: 211000/335864 (62.82%)
Progress: 221000/335864 (65.80%)
Progress: 231000/335864 (68.78%)
Progress: 241000/335864 (71.76%)
Progress: 251000/335864 (74.73%)
Progress: 261000/335864 (77.71%)
Progress: 271000/335864 (80.69%)
Progress: 281000/335864 (83.66%)

In [30]:
remaining_knn = cudf.DataFrame({
    "feature": knn_features,
    "missing_count": [
        int(df[col].isna().sum())
        for col in knn_features
    ]
})

remaining_knn["missing_percent"] = (
    remaining_knn["missing_count"] / len(df) * 100
)

remaining_knn = remaining_knn.sort_values(
    "missing_count",
    ascending=False
)

remaining_knn.head(30)

,feature,missing_count,missing_percent
0,V1,0,0.0
1,V2,0,0.0
2,V3,0,0.0
3,V4,0,0.0
4,V5,0,0.0
5,V6,0,0.0
6,V7,0,0.0
7,V8,0,0.0
8,V9,0,0.0
9,V10,0,0.0


In [31]:
total_missing = 0

for col in df.columns:

    total_missing += int(
        df[col].isna().sum()
    )

print(
    "Total remaining missing values:",
    total_missing
)

Total remaining missing values: 29363045


In [32]:
print("Final dataframe shape:", df.shape)

print("\nFirst 5 rows:")
print(df.head())

Final dataframe shape: (590540, 687)

First 5 rows:
   TransactionID  isFraud  TransactionDT  TransactionAmt ProductCD    card1  \
0      2988280.0      0.0       130016.0           50.00         W  17400.0   
1      2988281.0      0.0       130049.0          166.00         W   2616.0   
2      2988282.0      1.0       130050.0          117.00         W  11839.0   
3      2988283.0      0.0       130052.0           57.00         W   1939.0   
4      2988284.0      0.0       130056.0          209.95         W   1804.0   

   card2  card3       card4  card5  ... V330_missing  V331_missing  \
0  174.0  150.0        visa  226.0  ...            1             1   
1   <NA>  150.0    discover  102.0  ...            1             1   
2  490.0  150.0        visa  226.0  ...            1             1   
3  360.0  150.0        visa  166.0  ...            1             1   
4  161.0  150.0  mastercard  117.0  ...            1             1   

   V332_missing  V333_missing  V334_missing  V335_mi

In [33]:
print(
    "GPU memory used:",
    round(
        cp.get_default_memory_pool().used_bytes()
        / (1024 ** 3),
        3
    ),
    "GB"
)

print(
    "GPU memory pool:",
    round(
        cp.get_default_memory_pool().total_bytes()
        / (1024 ** 3),
        3
    ),
    "GB"
)

GPU memory used: 0.0 GB
GPU memory pool: 0.0 GB


In [ ]:
import os
import gc
import cupy as cp

OUTPUT_PATH = "path"
CHUNK_SIZE = 10000

print("Freeing GPU memory before CSV export...")

# Delete KNN temporary objects if they still exist
for var in [
    "knn",
    "scaler",
    "reference_scaled",
    "reference_gpu",
    "reference_mean",
    "reference_df",
    "reference_filled"
]:
    if var in globals():
        del globals()[var]

gc.collect()

# Free CuPy memory pool
cp.get_default_memory_pool().free_all_blocks()
cp.get_default_pinned_memory_pool().free_all_blocks()

print("Starting CSV export...")

# Remove old file if it exists
if os.path.exists(OUTPUT_PATH):
    os.remove(OUTPUT_PATH)

total_rows = len(df)

for start in range(0, total_rows, CHUNK_SIZE):
    end = min(start + CHUNK_SIZE, total_rows)

    print(f"Writing rows {start:,} - {end:,} / {total_rows:,}")

    # Move only this chunk from GPU → CPU
    chunk = df.iloc[start:end].to_pandas()

    # First chunk writes the header
    chunk.to_csv(
        OUTPUT_PATH,
        mode="w" if start == 0 else "a",
        header=(start == 0),
        index=False
    )

    del chunk
    gc.collect()

print("================================")
print("CSV export completed!")
print(f"Saved to: {OUTPUT_PATH}")
print("================================")

MemoryError: std::bad_alloc: out_of_memory: CUDA error (failed to allocate 2362160 bytes) at: include/rmm/mr/cuda_memory_resource.hpp:54: cudaErrorMemoryAllocation out of memory

In [ ]:
processed_df = cudf.read_parquet(
    OUTPUT_PATH
)

print(
    "Loaded processed dataset:",
    processed_df.shape
)

print(processed_df.head())